In [47]:
USE_LIVE_DATA = False

In [26]:
from pathlib import Path
from typing import Literal
from dataclasses import dataclass
import math
import datetime

from IPython.display import display, Markdown
from five_safes_tes_workbench.workbench import Workbench
from partialstats.partials import SumOfSquaresPartial, SumPartial
from partialstats.combiners import Combiner, mean_combiner, variance_combiner
from partialstats.combiners.scalar import sum_combiner
from omop_metadata_utils import DistributionCodesets, count_bar
from contingency_table_utils import ContingencyTable, read_contingency_table_from_json, aggregate_tables
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Full example using the 5s-TES workbench
[The 5s-TES workbench](https://github.com/federated-research/5S-TES-Workbench) provides a set of tools for interacting with Five Safes TES.
You provide the workbench your credentials for connecting to Five Safes TES, and it will configure your connection to the submission layer, including writing TES messages to your specification and collecting results.
The full details of how to use the workbench can be found in its README.
Here we will not focus on those details, but on how to use it to carry out an analysis.

For this example, the configuration is held in a `config.yml` file, as described in [the workbench](https://github.com/federated-research/5S-TES-Workbench/blob/main/example-config.yml).
The parameters in the `config.yml` are loaded like this:

In [45]:
if USE_LIVE_DATA:
    wb = Workbench()
    
    wb.validate(config_path="config.yml")

INFO | Template registered: 'hello_world'
INFO | Template registered: 'custom'
INFO | Template registered: 'simple_sql'
INFO | Template registered: 'bunny'
INFO | Validation successful
INFO | Config: project='DelphiDemo' tes_base_url='https://api.5s-tes.federated-research.com/' minio_sts_endpoint='https://api.minio.5s-tes.federated-research.com/sts' minio_endpoint='https://api.minio.5s-tes.federated-research.com/' minio_output_bucket='126104output' tres=['Nottingham TRE 01', 'Nottingham TRE 02']
INFO | Auth mode: AuthMode.CREDENTIALS


For this demonstration, if you have access to the University of Nottingham submission layer, the project configuration is:

- project: "DelphiDemo"
- tes_base_url: "https://api.5s-tes.federated-research.com"
- minio_sts_endpoint: "https://api.minio.5s-tes.federated-research.com/sts"
- minio_endpoint: "https://api.minio.5s-tes.federated-research.com"
- minio_output_bucket: "126104output"
- tres:
    - "Nottingham TRE 01"
    - "Nottingham TRE 02"

For authentication, you will need to either get an access token from the submission layer user interface, or ask your administrator for keycloak details.

## Discovery
For this demonstration we will pretend to be a researcher interested in potential links between high blood pressure and skin cancer.

First we will need to submit a task to collect information about the codes used in our datasets.
For this, we use the `wb.build_tes` set of methods.
These write TES messages for you, based on your inputs.
There is a template for submitting bunny tasks, which report on OMOP metadata.

In [3]:
if USE_LIVE_DATA:
    wb.build_tes.bunny(
        name="OMOP metadata",
        command=[
            "--body-json",
            '{"code":"GENERIC","analysis":"DISTRIBUTION","uuid":"123","collection":"test","owner":"me"}',
            "--output",
            "/outputs/output.json",
            "--no-encode",
        ],
    )

INFO | Building TES task from template: 'bunny'
INFO | Resolving template: 'bunny'
INFO | TES Task built successfully
INFO | TES payload:
{
   "name": "OMOP metadata",
   "description": "Bunny CLI analytics task",
   "outputs": [
      {
         "url": "s3://",
         "path": "/outputs",
         "type": "DIRECTORY",
         "name": "Query Results",
         "description": "Results from the requested query execution"
      }
   ],
   "executors": [
      {
         "image": "ghcr.io/health-informatics-uon/five-safes-tes-analytics-bunny-cli:1.6.0",
         "command": [
            "--body-json",
            "{\"code\":\"GENERIC\",\"analysis\":\"DISTRIBUTION\",\"uuid\":\"123\",\"collection\":\"test\",\"owner\":\"me\"}",
            "--output",
            "/outputs/output.json",
            "--no-encode"
         ]
      }
   ],
   "tags": {
      "project": "DelphiDemo",
      "tres": "Nottingham TRE 01|Nottingham TRE 02"
   },
   "creation_time": "2026-05-18T10:58:02.757673+00:00"

The `wb.submit()` method submits the last TES message built to your submission layer.
It will also tell you what ID your task was given.

In [4]:
if USE_LIVE_DATA:
    wb.submit()

INFO | Fetching token from keycloak...
INFO | Requesting keycloak token from https://drs-core-identity.azurewebsites.net/realms/Dare-Control/protocol/openid-connect/token
INFO | Keycloak token fetched successfully
INFO | Task submitted successfully! ID: 1241


'1241'

You can then use `wb.fetch_outputs()` to collect your outputs from the submission layer, once an output checker has approved them for egress.
By default, it will pick up the last task you submitted, which will have an ID that varies by the submission, and download the files to `./output/[TRE NAME]/[TASK ID]/`.

In [50]:
if USE_LIVE_DATA:
    metadata_paths = wb.fetch_outputs()
else:
    metadata_paths = {
    "Nottingham TRE 01": ["./output/Nottingham TRE 01/1242/output.json"],
    "Nottingham TRE 02": ["./output/Nottingham TRE 02/1243/output.json"],
}

This means we can now use the downloaded files for metadata visualisation, as described in the OMOP metadata notebook.

In [51]:
# unpack the metadata paths from the lists. Note that this works because there is only one file from each TRE.
metadata_paths = {k: v[0] for k, v in metadata_paths.items()}

codesets = DistributionCodesets(metadata_paths)

You can look at the raw tables from the query in the `.tables` attribute.

In [52]:
codesets.tables["Nottingham TRE 01"].head()

,TRE,BIOBANK,CODE,COUNT,ALTERNATIVES,DATASET,OMOP,OMOP_DESCR,CATEGORY
0,Nottingham TRE 01,test,OMOP:31967,310,NaN,NaN,31967,Nausea,Condition
1,Nottingham TRE 01,test,OMOP:75576,320,NaN,NaN,75576,Irritable bowel syndrome,Condition
2,Nottingham TRE 01,test,OMOP:77074,590,NaN,NaN,77074,Pain of joint,Condition
3,Nottingham TRE 01,test,OMOP:80180,120,NaN,NaN,80180,Osteoarthritis,Condition
4,Nottingham TRE 01,test,OMOP:80809,330,NaN,NaN,80809,Rheumatoid arthritis,Condition


You can get the counts for each code on each TRE with the `counts_by_TRE` property.

In [53]:
codesets.counts_by_TRE.head()

TRE,Nottingham TRE 01,Nottingham TRE 02
OMOP,,
8507,24720.0,24760.0
8516,6680.0,6750.0
8527,35440.0,35250.0
8532,25040.0,25010.0
8657,1830.0,1860.0


You can view how many codes your TREs have in common with the `code_intersections` property.

In [54]:
codesets.code_intersections

,0
"['Nottingham TRE 01', 'Nottingham TRE 02']",779
['Nottingham TRE 01'],17
['Nottingham TRE 02'],24


You can plot the $k$ codes with the highest counts using `.plot_top_k_by_count(k)`.
If you run this notebook, you can hover over the bars to get the OMOP description of that code.

In [55]:
codesets.plot_top_k_by_count(10)

alt.Chart(...)

If you have codes that you're interested in, you can use the `.plot_by_codes(list_of_codes)` method to get a barplot of those.

In [16]:
codesets.plot_by_codes([3004000, 2006977])

alt.Chart(...)

In [21]:
codesets.plot_by_codes(codesets.get_codes_by_membership("['Nottingham TRE 01']")["OMOP"])

alt.Chart(...)

As we're interested in skin cancer, we can look for codes matching either "cancer" or "neoplasm".

In [22]:
codesets.plot_by_codes(codesets.get_codes_by_substring_match("cancer|neoplasm")["OMOP"])

alt.Chart(...)

We can see a couple of thousand people have "Primary malignant neoplasm of skin", concept_id 139750.
Next, we can look at blood pressure.

In [23]:
codesets.plot_by_codes(codesets.get_codes_by_substring_match("pressure")["OMOP"])

alt.Chart(...)

We can see almost everyone in the dataset has had their systolic blood pressure measured (concept_id 3004249).

You can get a heatmap of how many codes are in each combination of datasets as a heatmap with `.plot_count_heatmap`

In [24]:
codesets.plot_count_heatmap().properties(width=600, height=600)

alt.Chart(...)

You can also get an [Upset plot](https://upset.app/) for your datasets.
This is like a Venn diagram, but instead of numbers written in circles, you get bars proportional to the number of codes present in each combination of TREs.
This shows the same information as the `code_insersections` property.
If you only have a couple of TREs, this isn't terribly useful, but once you have more than three, the number of combinations is much higher, and Venn diagrams get hard to read.

In [27]:
codesets.plot_upset()

alt.VConcatChart(...)